In [1]:
import pandas as pd 
import  re 


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Dense, Embedding ,LSTM
from tensorflow.keras.models import Sequential

In [ ]:
df= pd.read_csv(r"SMSSpamCollection",
            sep="\t",
           header=None,
           names=["label","message"])

In [3]:
df.columns

Index(['label', 'message'], dtype='str')

In [4]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
def preprocess(text):
    text=text.lower()
    text=re.sub(r"[^a-zA-Z\s]","",text)
    return text

In [6]:
df["message"]=df["message"].apply(preprocess)

In [7]:
le = LabelEncoder()
df["label"]=le.fit_transform(df["label"])


In [8]:
#x and y
x=df["message"]
y=df["label"]

In [9]:
#splitting
x_train,x_test,y_train,y_test=train_test_split(x,y,
                                              random_state=42,
                                              test_size=0.2)

In [10]:
#tokenization
tokens=Tokenizer(num_words=5000)
tokens.fit_on_texts(x_train)

x_train = tokens.texts_to_sequences(x_train)
x_test = tokens.texts_to_sequences(x_test)

In [12]:
#padding
x_train=pad_sequences(x_train,maxlen=100)
x_test =pad_sequences(x_test,maxlen=100)

In [13]:

model=Sequential([
    Embedding(5000,2),
    LSTM(32),
    Dense(1,activation="sigmoid")
])

In [14]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [15]:
#fit

history=model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size = 32
)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8694 - loss: 0.3575
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9518 - loss: 0.1601
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9751 - loss: 0.0909
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9825 - loss: 0.0644
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9850 - loss: 0.0532


In [18]:
loss,accuracy=model.evaluate(x_test,y_test)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9794 - loss: 0.0694


In [20]:
print("Accuracy :",accuracy*100)

Accuracy : 97.93722033500671
